In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
# 1. load encoder (already loaded from pretrained BERT)
# 2. load decoder from stage 1
checkpoint = torch.load("stage1_best.pt", map_location=device)
decoder.load_state_dict(checkpoint["decoder"])
for param in decoder.parameters():
    param.requires_grad = False
print("stage1 decoder loaded and frozen")

# 3. init new stage 2 components fresh
pooler     = SequencePooler(hidden_dim=768, latent_dim=256).to(device)
metric_net = MetricNet(latent_dim=256).to(device)
flow_net   = FlowNet(latent_dim=256).to(device)

# 4. optimizer only touches stage 2 components
optimizer = AdamW(
    list(pooler.parameters()) +
    list(metric_net.parameters()) +
    list(flow_net.parameters()),
    lr=1e-4
)

FileNotFoundError: [Errno 2] No such file or directory: 'stage1_best.pt'

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MetricNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim * latent_dim)
        )
    
    def forward(self, z):
        B = z.shape[0]
        L_flat = self.net(z)                                    # [B, d*d]
        L = L_flat.view(B, self.latent_dim, self.latent_dim)   # [B, d, d]
        
        # enforce lower triangular
        L = torch.tril(L)
        
        # enforce positive diagonal via softplus
        diag_idx = torch.arange(self.latent_dim)
        L = L.clone()
        L[:, diag_idx, diag_idx] = F.softplus(L[:, diag_idx, diag_idx]) + 1e-4
        
        # SPD matrix
        G = L @ L.transpose(-2, -1)                            # [B, d, d]
        return G, L


# ── test ──────────────────────────────────────────────────────────────
metric_net = MetricNet(latent_dim=256).to(device)

z_test = torch.randn(2, 256).to(device)
G, L = metric_net(z_test)

eigenvalues = torch.linalg.eigvalsh(G)

print(f"G shape:        {G.shape}")           # [2, 256, 256]
print(f"L shape:        {L.shape}")           # [2, 256, 256]
print(f"min eigenvalue: {eigenvalues.min().item():.6f}")  # must be > 0
print(f"max eigenvalue: {eigenvalues.max().item():.6f}")
print(f"SPD check:      {'PASSED' if eigenvalues.min() > 0 else 'FAILED'}")

G shape:        torch.Size([2, 256, 256])
L shape:        torch.Size([2, 256, 256])
min eigenvalue: 0.009496
max eigenvalue: 3.867041
SPD check:      PASSED


In [4]:
class FlowNet(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 512),  # +1 for time t
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, latent_dim)
        )
    
    def forward(self, z_t, t):
        # z_t: [B, latent_dim]
        # t:   [B]
        t_expand = t.unsqueeze(-1)                  # [B, 1]
        inp      = torch.cat([z_t, t_expand], dim=-1)  # [B, latent_dim+1]
        return self.net(inp)                        # [B, latent_dim]


# ── test ──────────────────────────────────────────────────────────────
flow_net = FlowNet(latent_dim=256).to(device)

z_t_test = torch.randn(2, 256).to(device)
t_test   = torch.rand(2).to(device)

v_pred = flow_net(z_t_test, t_test)

print(f"z_t shape: {z_t_test.shape}")  # [2, 256]
print(f"t shape:   {t_test.shape}")    # [2]
print(f"v_pred shape: {v_pred.shape}") # [2, 256]
print("FlowNet check: PASSED" if v_pred.shape == z_t_test.shape else "FAILED")

z_t shape: torch.Size([2, 256])
t shape:   torch.Size([2])
v_pred shape: torch.Size([2, 256])
FlowNet check: PASSED
